# పాఠం 10 - ఉత్పత్తిలో AI ఏజెంట్లు

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్స్ కోసం **ఉత్పత్తి నమూనాలు** నేర్చుకుంటారు. మేము కవర్ చేస్తాము:

- **Observability** — ఏజెంట్ పరస్పర చర్యలకు టైమింగ్ మరియు లాగింగ్ జోడించడం
- **Evaluation** — ప్రతిస్పందన నాణ్యతను స్కోర్ చేయడానికి ఒక మూల్యాంకన ఏజెంట్‌ను ఉపయోగించడం
- **Cost management** — టోకెన్ ఆప్టిమైజేషన్ మరియు మోడల్ ఎంపిక కోసం వ్యూహాలు

సన్నివేశం ఒక **ప్రయాణ ఏజెంట్** ఇది వినియోగదారులకు ప్రయాణాలు ప్లాన్ చేయడంలో సహాయపడుతుంది, పైన మానిటరింగ్ మరియు మూల్యాంకన పొరలు ఉన్నవి.


## సెటప్


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ఉత్పత్తిలో పరిగణించవలసిన అంశాలు

AI ఏజెంట్లను ప్రోటోటైప్స్ నుండి ఉత్పత్తికి మార్చేటప్పుడు మూడు ప్రధాన స్తంభాలపై జాగ్రత్తగా దృష్టి పెట్టాలి:

1. **పర్యవేక్షణ సామర్థ్యం** — ఏజెంట్ ఏమి చేస్తున్నదో, అది ఎంత సమయం తీసుకుంటుందో, మరియు ఇది ఏ సాధనాలను పిలుస్తుందో మీకు స్పష్టంగా కనిపించాలి. ట్రేసింగ్ మరియు లాగింగ్ లేకుండా, ఉత్పత్తి సమస్యలను డీబగ్ చేయడం దాదాపు అసాధ్యం.

2. **మూల్యాంకనం** — స్వయంచాలక నాణ్యత తనిఖీలు కాలానుగుణంగా ఏజెంట్ యొక్క ప్రతిస్పందనలు ఖచ్చితంగా, సంపూర్ణంగా మరియు సహాయకంగా ఉండేలా నిర్ధారిస్తాయి. ఒక మూల్యాంకక ఏజెంట్ నిర్వచించిన ప్రమాణాల ప్రకారం ప్రతిస్పందనలకు స్కోరు ఇవ్వగలదు.

3. **ఖర్చు నిర్వహణ** — టోకెన్ వినియోగం నేరుగా ఖర్చులపై ప్రభావం చూపుతుంది. ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక మరియు కాషింగ్ వంటి వ్యూహాలు నాణ్యతను బలహీనపరచకుండా ఖర్చులను నియంత్రణలో ఉంచగలవు.


## ఒక పరిశీలించదగిన ఏజెంట్‌ను నిర్మించడం

మేము ప్రయాణ సాధనాలను నిర్వచించి, ఏజెంట్ కాల్‌ను టైమింగ్‌తో ర్యాప్ చేసి తద్వారా మేము విలంబాన్ని పర్యవేక్షించగలుగుతాము. ఉత్పత్తిలో మీరు OpenTelemetry లేదా ఇలాంటి ట్రేసింగ్ బ్యాక్‌ఎండ్‌తో సమగ్రీకరించవచ్చు.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## మూల్యాంకన నమూనాలు

సాధారణ ఉత్పత్తి నమూనా రెండవ ఏజెంట్‌ను **మూల్యాంకకుడు** గా ఉపయోగించడం. మూల్యాంకకుడు ప్రధాన ఏజెంట్ యొక్క ప్రతిస్పందనను పూర్ణత, ఖచ్చితత్వం, మరియు సహాయకత్వం వంటి ముందుగా నిర్వచించబడిన ప్రమాణాలపై స్కోర్ చేస్తాడు.

దీనివల్ల సాధ్యమవుతుంది:
- వినియోగదారులకు ప్రతిస్పందనలు చేరడానికి ముందు ఆటోమేటెడ్ నాణ్యత గేట్లు
- ప్రాంప్ట్‌లు లేదా మోడళ్లు మారినప్పుడు రెగ్రెషన్ గుర్తింపు
- కాలంతో పాటు ఏజెంట్ పనితీరు యొక్క నిరంతర పర్యవేక్షణ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ఖర్చు నిర్వహణ వ్యూహాలు

ఉత్పత్తి AI ఏజెంట్స్ కోసం ఖర్చులను నియంత్రించడం అత్యవసరం. ఇక్కడ ముఖ్యమైన వ్యూహాలు ఉన్నాయి:

| Strategy | Description |
|---|---|
| **ప్రాంప్ట్ ఆప్టిమైజేషన్** | సిస్టమ్ సూచనలను సంక్షిప్తంగా ఉంచండి. ఇన్పుట్ టోకన్లను తగ్గించడానికి పునరావృతమైన సందర్భాన్ని తీసివేయండి. |
| **మోడల్ ఎంపిక** | వర్గీకరణ లేదా ఎక్స్‌ట్రాక్షన్ వంటి సరళ పనుల కోసం చిన్న, తక్కువ ఖర్చు గల మోడెల్స్ (ఉదా. GPT-4o-mini) ఉపయోగించండి, మరియు క్లిష్టమైన తర్కాల కోసం పెద్ద సామర్థ్యపు మోడల్స్‌ను మాత్రమే आरక్షించండి. |
| **క్యాచింగ్** | టూల్ ఫలితాలు మరియు తరచుగా వచ్చే ప్రశ్నలను క్యాష్ చేయండి, పునరావృత API కాల్స్ నివారించడానికి. |
| **టోకెన్ బడ్జెట్స్** | అనుకోని పొడవైన ప్రతిస్పందనలను నివారించడానికి `max_tokens` పరిమితులను సెట్ చేయండి. |
| **బ్యాచ్ చేయడం** | సాధ్యమైనప్పుడు బహుళ వినియోగదారు ప్రశ్నలను ఒకే API కాల్‌లో సమూహపరచండి. |

ప్రాయోగికంగా, ఒక స్థాయిగత విధానం బాగా పనిచేస్తుంది: సరళ అభ్యర్థనలను వేగవంతమైన, తక్కువ ఖర్చు గల మోడల్‌కి దారితీసి, కేవలం క్లిష్టమైన ప్రశ్నలను మాత్రమే అధిక సామర్థ్య గల మోడల్‌కు ఎస్కలేట్ చేయండి.


## సారాంశం

ఈ పాఠంలో మీరు ఎలా చేయాలో నేర్చుకున్నారు:

1. **పర్యవేక్షణను చేర్చడం** ఎజెంట్ ఇంటరాక్షన్లలో టైమింగ్ మరియు లాగింగ్‌తో, ట్రేసింగ్ మరియు మానిటరింగ్ కోసం పునాదిని సిద్ధం చేయడం.
2. **ఎజెంట్ స్పందనలను మూల్యాంకనం చేయడం** స్వయంచాలకంగా ఒక మూల్యాంకన ఏజెంట్ ఉపయోగించి సంపూర్ణత, ఖచ్చితత్వం మరియు సహాయకత్వాన్ని స్కోర్ చేయడం.
3. **ఖర్చులను నిర్వహించడం** ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, క్యాషింగ్, మరియు టోకెన్ బడ్జెట్ల ద్వారా.

ఈ ఉత్పత్తి నమూనాలు మీ AI ఏజెంట్లను విశ్వసనీయంగా, కొలవదగినలా మరియు స్కేల్‌లో ఖర్చు-ప్రభావవంతంగా ఉండటానికి సహాయపడతాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
నిరాకరణ:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ద్వారా అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, స్వయంచాలక అనువాదాల్లో తప్పులు లేదా లోపాలు ఉండవచ్చు. మూల పత్రాన్ని దాని మాతృభాషలో ఉన్న ప్రతిని అధికారం కలిగిన మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి వృత్తిపరమైన మానవ అనువాదం చేయించుకోవాలని సూచించబడుతుంది. ఈ అనువాదాన్ని ఉపయోగించడం వలన సంభవించే ఏవైనా అపార్థాలు లేదా తప్పుదారులకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
